## What is Delta Lake?

Plain data lakes store files — Parquet, JSON, CSV — in object storage like S3 or ADLS. They are cheap and scalable, but they have no built-in mechanism for atomicity, consistency, or schema enforcement. A failed write leaves partial data behind. Two concurrent writes corrupt each other. There is no way to undo a mistake.

**Delta Lake** is an open-source storage layer that adds a **transaction log** on top of Parquet files. That log is what makes a Delta table different from a folder of Parquet files:

```
Plain Parquet lake                   Delta Lake
──────────────────────               ─────────────────────────────────
s3://bucket/transactions/            s3://bucket/transactions/
  part-0001.parquet                    _delta_log/
  part-0002.parquet                      00000000000000000000.json  ← transaction 0
  part-CORRUPT.parquet  ← ???            00000000000000000001.json  ← transaction 1
  (no history, no undo)                  00000000000000000002.json  ← transaction 2
                                       part-0001.parquet
                                       part-0002.parquet
                                         (every file accounted for in the log)
```

In this notebook you will learn:
- The problems with plain Parquet lakes that Delta Lake solves
- The **transaction log** (`_delta_log`) — what it contains and how it works
- Creating, reading, and writing Delta tables
- **ACID transactions** — atomicity, consistency, isolation, durability in practice
- **Schema enforcement** — Delta rejects data that does not match the table schema
- **Schema evolution** — safely adding columns with `mergeSchema`
- **Time travel** — querying a table as it was at a past version or timestamp
- **Table history** — auditing every operation with `DESCRIBE HISTORY`
- Converting an existing Parquet table to Delta

In [ ]:
import os, json, shutil
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, lit,
    current_timestamp, to_date, expr
)

spark = (
    SparkSession.builder
    .appName("WhatIsDeltaLake")
    .config("spark.sql.extensions",
            "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog",
            "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE    = os.path.dirname(os.path.abspath("18-what-is-delta-lake.ipynb"))
DATA    = os.path.join(BASE, "data")
SCRATCH = os.path.join(BASE, "_scratch_18")

os.makedirs(SCRATCH, exist_ok=True)

print(f"Spark {spark.version} ready")

### The Problems with Plain Data Lakes

Object storage (S3, ADLS, GCS) is cheap and infinitely scalable. Parquet is an efficient columnar format. Together they make a compelling foundation — but without a transaction layer, several painful failure modes emerge:

**1. No atomicity — partial writes**
```
Job writes 100 files.  Cluster crashes after 60 files.
Result: 60 files committed, 40 missing.  Readers see incomplete data.
No way to know which rows are present without re-processing everything.
```

**2. No isolation — concurrent write corruption**
```
Job A overwrites partition date=2024-01-01.
Job B reads partition date=2024-01-01 at the same time.
Job B sees a mix of old and new files — inconsistent snapshot.
```

**3. No schema enforcement**
```
Upstream pipeline adds a new column with a wrong data type.
New files land in the folder.  Existing readers break at runtime.
No gate at write time to catch the mismatch.
```

**4. No versioning — mistakes are permanent**
```
df.write.mode("overwrite").parquet(path)
# Wrong filter. Data gone. No undo.
```

**5. Slow metadata queries ("listing problem")**
```
SELECT count(*) FROM transactions WHERE date = '2024-01-01'
→ S3 LIST call for every partition directory → thousands of API calls → minutes of latency
```

Delta Lake solves all five with a single addition: the **transaction log**.

### The Transaction Log (`_delta_log`)

Every Delta table has a `_delta_log/` directory alongside its data files. This directory is the source of truth for the entire table.

**What the log contains:**
- A sequence of JSON files, one per committed transaction: `00000000000000000000.json`, `00000000000000000001.json`, …
- Periodically, a **checkpoint** Parquet file that compresses the full log state into a single file (every 10 commits by default)
- The log records which data files are **added** and which are **removed** in each transaction

**What a single log entry looks like:**
```json
{ "commitInfo": { "operation": "WRITE", "timestamp": 1704067200000, "operationParameters": {...} } }
{ "add": { "path": "part-00000-abc.parquet", "size": 204800, "stats": "{\"numRecords\": 1000}" } }
{ "add": { "path": "part-00001-def.parquet", "size": 198400, "stats": "{\"numRecords\": 987}"  } }
```

**How readers use the log:**
1. Read the latest checkpoint (if any) to get the table state up to that point
2. Read the subsequent JSON log files to get the latest transactions
3. Compute the set of **active** data files = all `add` entries minus all `remove` entries
4. Read only those files — no directory listing required

This is why Delta reads are fast even on huge tables: instead of listing thousands of files, Spark reads the compact log and knows exactly which files to open.

### Load the Bank Data

The examples below use the datasets created by `generate_bank_data.ipynb`.

In [ ]:
customers = spark.read.option("header", "true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),      False),
    StructField("card_id",     StringType(),      False),
    StructField("customer_id", StringType(),      False),
    StructField("amount",      DecimalType(18,2), False),
    StructField("merchant",    StringType(),      True),
    StructField("category",    StringType(),      True),
    StructField("txn_ts",      TimestampType(),   False),
    StructField("status",      StringType(),      False),
    StructField("is_fraud",    BooleanType(),     False),
])).parquet(f"{DATA}/card_transactions")

loan_accounts = spark.read.option("multiLine", "true").schema(StructType([
    StructField("loan_id",       StringType(),      False),
    StructField("customer_id",   StringType(),      False),
    StructField("loan_type",     StringType(),      False),
    StructField("principal",     DecimalType(18,2), False),
    StructField("interest_rate", DoubleType(),      False),
    StructField("tenure_months", IntegerType(),     False),
    StructField("disbursed_on",  DateType(),        False),
    StructField("status",        StringType(),      False),
])).json(f"{DATA}/loan_accounts")

print(f"customers  : {customers.count():>5} rows")
print(f"card_txns  : {card_txns.count():>5} rows")
print(f"loans      : {loan_accounts.count():>5} rows")

### Creating a Delta Table

Writing to Delta is identical to writing Parquet — just change the format string. Spark creates the data files **and** the `_delta_log/` directory automatically.

In [ ]:
CUSTOMERS_DELTA = os.path.join(SCRATCH, "customers_delta")

# Write customers as a Delta table
(
    customers
    .write
    .format("delta")
    .mode("overwrite")
    .save(CUSTOMERS_DELTA)
)

print("Delta table written. Directory contents:")
for entry in sorted(os.listdir(CUSTOMERS_DELTA)):
    print(f"  {entry}")

print("\n_delta_log contents:")
log_dir = os.path.join(CUSTOMERS_DELTA, "_delta_log")
for entry in sorted(os.listdir(log_dir)):
    size = os.path.getsize(os.path.join(log_dir, entry))
    print(f"  {entry}  ({size} bytes)")

In [ ]:
# Peek inside transaction 0 — the initial write
log_file = os.path.join(log_dir, "00000000000000000000.json")
with open(log_file) as f:
    lines = f.readlines()

print(f"Transaction log has {len(lines)} entries.\n")

for line in lines:
    entry = json.loads(line)
    key   = list(entry.keys())[0]
    if key == "commitInfo":
        print(f"commitInfo  operation : {entry['commitInfo'].get('operation')}")
    elif key == "metaData":
        schema_str = entry["metaData"].get("schemaString", "")
        print(f"metaData    schema columns: {schema_str[:80]}...")
    elif key == "add":
        stats = json.loads(entry["add"].get("stats", "{}"))
        print(f"add         {entry['add']['path'][-40:]}  rows={stats.get('numRecords','?')}")
    elif key == "protocol":
        print(f"protocol    minReaderVersion={entry['protocol']['minReaderVersion']}  "
              f"minWriterVersion={entry['protocol']['minWriterVersion']}")

### Reading a Delta Table

Reading Delta is the same as reading any other format. Under the hood Spark consults the transaction log to find the current set of active files — no directory listing required.

In [ ]:
# Read via path
cust_delta = spark.read.format("delta").load(CUSTOMERS_DELTA)

print("Schema:")
cust_delta.printSchema()

print(f"Row count : {cust_delta.count()}")
cust_delta.select("customer_id", "full_name", "city", "kyc_verified").show(5)

In [ ]:
# Register as a SQL table and query with Spark SQL
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS customers_delta
    USING delta
    LOCATION '{CUSTOMERS_DELTA}'
""")

spark.sql("""
    SELECT city, count(*) AS customers
    FROM   customers_delta
    GROUP  BY city
    ORDER  BY customers DESC
    LIMIT  8
""").show()

### Writing to a Delta Table — append and overwrite

Delta supports the same `mode` options as any Spark write, but with ACID guarantees:

| Mode | Behaviour | Atomicity |
|---|---|---|
| `append` | Add new rows; existing rows untouched | Yes — all new files committed or none |
| `overwrite` | Replace the entire table | Yes — new files committed before old ones are removed |
| `overwrite` + `replaceWhere` | Replace only rows matching a condition | Yes — partition-level overwrite |
| `error` (default) | Fail if the table exists | — |
| `ignore` | No-op if the table exists | — |

Each write is a **new transaction** — a new JSON entry in the log. If the write fails mid-way, none of the new files appear in the log and the table is unchanged.

In [ ]:
TXN_DELTA = os.path.join(SCRATCH, "card_txns_delta")

# Initial write — version 0
first_half = card_txns.filter(col("txn_ts") < "2024-04-01")
(
    first_half
    .write
    .format("delta")
    .partitionBy("category")   # partition by category for efficient predicate pushdown
    .mode("overwrite")
    .save(TXN_DELTA)
)
print(f"Version 0: {spark.read.format('delta').load(TXN_DELTA).count()} rows")

# Append second half — version 1
second_half = card_txns.filter(col("txn_ts") >= "2024-04-01")
(
    second_half
    .write
    .format("delta")
    .partitionBy("category")
    .mode("append")
    .save(TXN_DELTA)
)
print(f"Version 1: {spark.read.format('delta').load(TXN_DELTA).count()} rows (after append)")

In [ ]:
# replaceWhere — overwrite only SUCCESS transactions in TRAVEL category
# All other categories + statuses remain untouched
replacement = (
    card_txns
    .filter((col("category") == "TRAVEL") & (col("status") == "SUCCESS"))
    .withColumn("status", lit("VERIFIED"))   # simulate a status update
)

(
    replacement
    .write
    .format("delta")
    .partitionBy("category")
    .mode("overwrite")
    .option("replaceWhere", "category = 'TRAVEL'")
    .save(TXN_DELTA)
)
print("Version 2: replaceWhere TRAVEL written")

# Confirm only TRAVEL rows were replaced
spark.read.format("delta").load(TXN_DELTA) \
    .groupBy("category", "status").count() \
    .orderBy("category", "status").show()

### ACID Transactions in Practice

Delta Lake guarantees all four ACID properties:

**Atomicity** — a write either commits all its files or commits nothing. If the job crashes after writing 60 of 100 files, those 60 files exist on disk but are never referenced in the log. Readers never see them. The table remains in its previous state.

**Consistency** — schema enforcement (see next section) ensures every committed row conforms to the declared schema. The table is always in a valid state.

**Isolation** — Delta uses **optimistic concurrency control**. Each writer reads the current table version, makes its changes, then attempts to commit. If another writer committed a conflicting change in the meantime, the commit fails and the writer can retry. Readers always see a consistent snapshot regardless of concurrent writes.

**Durability** — once the log JSON file is written to object storage, the transaction is permanent. Object storage itself provides the durability guarantee.

In [ ]:
# Atomicity demo: simulate a failed write
# In a real cluster, a failed job would leave orphan files that never appear in the log.
# Here we verify that the table count is stable across writes.

ATOMIC_DELTA = os.path.join(SCRATCH, "atomic_demo")

# Version 0: write 100 customers
sample_100 = customers.limit(100)
sample_100.write.format("delta").mode("overwrite").save(ATOMIC_DELTA)

count_before = spark.read.format("delta").load(ATOMIC_DELTA).count()
print(f"Before write attempt : {count_before} rows")

# Simulate a partial write by writing files manually but never committing to the log
# (Delta only reads what the log references — orphan files are invisible)
orphan_path = os.path.join(ATOMIC_DELTA, "orphan_file.parquet")
customers.limit(50).write.mode("overwrite").parquet(orphan_path)

count_after = spark.read.format("delta").load(ATOMIC_DELTA).count()
print(f"After orphan file    : {count_after} rows  ← unchanged, orphan file is invisible to Delta")
print(f"Orphan file exists   : {os.path.exists(orphan_path)}")

### Schema Enforcement

Delta Lake **rejects writes that do not match the table schema**. This is enforced at write time — not at read time — so data quality problems are caught at the source rather than downstream.

Schema enforcement checks:
- New columns not present in the table schema → **rejected** (unless `mergeSchema` is enabled)
- Missing columns that are not nullable → **rejected**
- Wrong data type for an existing column → **rejected**
- Extra columns with the same name but different type → **rejected**

In [ ]:
from pyspark.sql.utils import AnalysisException

ENFORCE_DELTA = os.path.join(SCRATCH, "schema_enforce")

# Establish the table with the known customer schema
customers.write.format("delta").mode("overwrite").save(ENFORCE_DELTA)
print(f"Table created. Schema:")
spark.read.format("delta").load(ENFORCE_DELTA).printSchema()

# Attempt to append a DataFrame with an extra column — should be REJECTED
customers_extra = customers.withColumn("credit_score", lit(750))
try:
    customers_extra.write.format("delta").mode("append").save(ENFORCE_DELTA)
    print("Write succeeded (unexpected)")
except AnalysisException as e:
    print("Write REJECTED by schema enforcement:")
    print(f"  {str(e)[:200]}")

In [ ]:
# Attempt to append with a wrong data type — should also be REJECTED
from pyspark.sql.functions import col

# Cast kyc_verified (BooleanType) to StringType — type mismatch
customers_wrong_type = customers.withColumn("kyc_verified", col("kyc_verified").cast("string"))

try:
    customers_wrong_type.write.format("delta").mode("append").save(ENFORCE_DELTA)
    print("Write succeeded (unexpected)")
except Exception as e:
    print("Write REJECTED — data type mismatch:")
    print(f"  {str(e)[:200]}")

# Confirm the table is unchanged
print(f"\nTable row count unchanged: {spark.read.format('delta').load(ENFORCE_DELTA).count()}")

### Schema Evolution — Adding Columns with mergeSchema

Schema enforcement blocks accidental schema changes. But sometimes you **intentionally** want to add a new column to an existing table — for example, when a new data source starts producing an additional field.

Enable `mergeSchema` to allow additive schema changes:
- New columns in the incoming data are **added** to the table schema
- Existing rows get `null` for the new column
- Existing columns are not removed
- Type changes (e.g., `int` → `string`) are still rejected unless the types are compatible

`mergeSchema` is an **opt-in** safety valve — you must explicitly request it for each write.

In [ ]:
EVOLVE_DELTA = os.path.join(SCRATCH, "schema_evolve")

# Version 0: base customers schema
customers.write.format("delta").mode("overwrite").save(EVOLVE_DELTA)
print("Version 0 columns:", spark.read.format("delta").load(EVOLVE_DELTA).columns)

# Version 1: new upstream data includes credit_score and risk_tier
customers_v2 = (
    customers
    .withColumn("credit_score", (700 + (col("customer_id").substr(5, 4).cast("int") % 200)).cast("int"))
    .withColumn("risk_tier",    lit("STANDARD"))
)

# mergeSchema=true → new columns added to the table schema
(
    customers_v2
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .save(EVOLVE_DELTA)
)
print("Version 1 columns:", spark.read.format("delta").load(EVOLVE_DELTA).columns)

# Old rows have null for the new columns
evolved = spark.read.format("delta").load(EVOLVE_DELTA)
print(f"\nTotal rows : {evolved.count()}")
print("\nRows where credit_score is null (original rows):")
evolved.filter(col("credit_score").isNull()) \
    .select("customer_id", "full_name", "credit_score", "risk_tier").show(3)
print("Rows where credit_score is not null (new rows):")
evolved.filter(col("credit_score").isNotNull()) \
    .select("customer_id", "full_name", "credit_score", "risk_tier").show(3)

### Time Travel — Querying Past Versions

Because every transaction is preserved in the log, Delta Lake lets you query the table **as it was at any point in time** — by version number or by timestamp. This is called **time travel**.

Time travel enables:
- **Audit queries** — "what did this table look like before yesterday's bad write?"
- **Rollback** — read a past version and overwrite the current table with it
- **Reproducibility** — training data snapshots for ML models
- **Debugging** — compare two versions to find what changed

The next notebook (ACID & Time Travel) covers the full API including rollback and retention settings. Here is the basic syntax.

In [ ]:
# Time travel by version number
v0 = spark.read.format("delta").option("versionAsOf", 0).load(TXN_DELTA)
v1 = spark.read.format("delta").option("versionAsOf", 1).load(TXN_DELTA)
v2 = spark.read.format("delta").load(TXN_DELTA)  # current (version 2)

print(f"Version 0 rows : {v0.count()}")
print(f"Version 1 rows : {v1.count()}")
print(f"Version 2 rows : {v2.count()} (current)")

# Confirm the TRAVEL category status difference between versions
print("\nTRAVEL status distribution:")
print("  Version 0:", v0.filter(col("category") == "TRAVEL").groupBy("status").count().collect())
print("  Version 2:", v2.filter(col("category") == "TRAVEL").groupBy("status").count().collect())

In [ ]:
# Time travel by timestamp — useful when you know when a bad write happened
from datetime import datetime, timezone

# Find the timestamps of the first two commits from the log
commit_timestamps = []
txn_log = os.path.join(TXN_DELTA, "_delta_log")
for fname in sorted(os.listdir(txn_log)):
    if fname.endswith(".json"):
        with open(os.path.join(txn_log, fname)) as f:
            for line in f:
                entry = json.loads(line)
                if "commitInfo" in entry:
                    ts_ms = entry["commitInfo"].get("timestamp", 0)
                    commit_timestamps.append((fname, ts_ms))

for fname, ts in commit_timestamps:
    dt = datetime.fromtimestamp(ts / 1000, tz=timezone.utc)
    print(f"  {fname}  →  {dt.isoformat()}")

# Read as of a specific timestamp (the moment after the first commit)
if commit_timestamps:
    ts_after_v0 = commit_timestamps[0][1] + 1000  # 1 second after version 0
    ts_str = datetime.fromtimestamp(ts_after_v0 / 1000, tz=timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
    try:
        v_by_ts = spark.read.format("delta").option("timestampAsOf", ts_str).load(TXN_DELTA)
        print(f"\nRows as of {ts_str}: {v_by_ts.count()}")
    except Exception as e:
        print(f"Timestamp query result: {str(e)[:150]}")

### Table History — DESCRIBE HISTORY

`DESCRIBE HISTORY` (SQL) or `DeltaTable.history()` (Python) returns an audit trail of every operation committed to the table — who ran it, when, what operation, how many files were added or removed, and what parameters were used.

This is useful for:
- Understanding what changed between two versions
- Finding the timestamp of a bad write before a rollback
- Compliance and data lineage auditing

In [ ]:
from delta.tables import DeltaTable

dt = DeltaTable.forPath(spark, TXN_DELTA)

history = dt.history()
history.select(
    "version",
    "timestamp",
    "operation",
    "operationParameters",
    col("operationMetrics.numFiles").alias("numFiles"),
    col("operationMetrics.numOutputRows").alias("numOutputRows"),
).show(truncate=False)

In [ ]:
# Same via Spark SQL — register the table first
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS card_txns_delta
    USING delta
    LOCATION '{TXN_DELTA}'
""")

spark.sql("DESCRIBE HISTORY card_txns_delta") \
    .select("version", "timestamp", "operation", "operationParameters") \
    .show(truncate=False)

### Delta Table Details and Stats

`DESCRIBE DETAIL` returns the current table properties: format, location, number of files, size in bytes, partition columns, and table properties. This is the quickest way to confirm a table's state without reading all its data.

In [ ]:
detail = spark.sql("DESCRIBE DETAIL card_txns_delta")
detail.select(
    "format", "location", "partitionColumns",
    "numFiles", "sizeInBytes"
).show(truncate=False)

# Equivalent using the DeltaTable API
dt_detail = DeltaTable.forPath(spark, TXN_DELTA).detail()
row = dt_detail.collect()[0]
print(f"\nFormat            : {row['format']}")
print(f"Num files         : {row['numFiles']}")
print(f"Size (bytes)      : {row['sizeInBytes']:,}")
print(f"Partition columns : {row['partitionColumns']}")

### Converting Parquet to Delta

If you already have a Parquet table, you can convert it to Delta **in place** without rewriting the data files. The `CONVERT TO DELTA` command generates a `_delta_log/` directory by scanning the existing Parquet files and recording them as the initial transaction.

This is a zero-copy operation — no data is moved or rewritten. The conversion is fast and reversible (Delta can be converted back to Parquet by removing the `_delta_log/` directory, though all history is lost).

**Limitations:** the Parquet files must not use Hive-style partitioning with types that Delta does not support. Provide the partition schema if the table is partitioned.

In [ ]:
PARQUET_PATH = os.path.join(SCRATCH, "loans_parquet")
CONVERT_PATH = os.path.join(SCRATCH, "loans_converted")

# Write a plain Parquet table (no Delta)
loan_accounts.write.mode("overwrite").partitionBy("loan_type").parquet(PARQUET_PATH)

# Verify no _delta_log yet
print("Before conversion — directory contents:")
for e in sorted(os.listdir(PARQUET_PATH)):
    print(f"  {e}")

# Copy so we can demonstrate conversion without modifying the original
shutil.copytree(PARQUET_PATH, CONVERT_PATH)

# Convert to Delta — partition schema must match
spark.sql(f"""
    CONVERT TO DELTA parquet.`{CONVERT_PATH}`
    PARTITIONED BY (loan_type STRING)
""")

print("\nAfter conversion — directory contents:")
for e in sorted(os.listdir(CONVERT_PATH)):
    print(f"  {e}")

# Read the converted table as Delta
converted = spark.read.format("delta").load(CONVERT_PATH)
print(f"\nConverted table rows: {converted.count()}")
converted.groupBy("loan_type").count().orderBy("count").show()

### Delta vs Plain Parquet — Side-by-Side Comparison

| Capability | Plain Parquet | Delta Lake |
|---|---|---|
| ACID writes | No | Yes — transaction log |
| Atomic multi-file commit | No | Yes |
| Concurrent write isolation | No | Yes — optimistic concurrency |
| Schema enforcement | No | Yes — at write time |
| Schema evolution | Manual | `mergeSchema` option |
| Time travel | No | Yes — version or timestamp |
| Audit history | No | `DESCRIBE HISTORY` |
| Upserts (MERGE) | No | Yes — `MERGE INTO` |
| Deletes | No (rewrite partition) | Yes — `DELETE FROM` |
| Updates | No (rewrite partition) | Yes — `UPDATE SET` |
| File compaction | Manual | `OPTIMIZE` command |
| Data skipping | Partition pruning only | Column stats in log → row-group skipping |
| Streaming source/sink | Append only, fragile | Full streaming support, exactly-once |
| Convert from Parquet | — | `CONVERT TO DELTA` (zero-copy) |

In [ ]:
# Quick benchmark: read performance with data skipping
# Delta stores min/max stats per column per file in the log.
# A filter on a non-partition column skips entire files without opening them.

# Write card_txns as both plain Parquet and Delta (no partitioning, to show column stats)
PLAIN_PATH = os.path.join(SCRATCH, "txns_plain")
DELTA_PATH = os.path.join(SCRATCH, "txns_stats")

card_txns.write.mode("overwrite").parquet(PLAIN_PATH)
card_txns.write.format("delta").mode("overwrite").save(DELTA_PATH)

import time

# Filter on amount (non-partition column) — Delta uses log stats to skip files
pred = col("amount") > 590

t0 = time.time()
n_parquet = spark.read.parquet(PLAIN_PATH).filter(pred).count()
t_parquet = time.time() - t0

t0 = time.time()
n_delta = spark.read.format("delta").load(DELTA_PATH).filter(pred).count()
t_delta = time.time() - t0

print(f"Filter amount > 590:")
print(f"  Plain Parquet : {n_parquet} rows  in {t_parquet:.2f}s")
print(f"  Delta Lake    : {n_delta} rows  in {t_delta:.2f}s")
print("  (Delta skips files whose max(amount) < 590 using log statistics)")

### Summary

**Delta Lake** is an open-source storage layer that adds a **transaction log** on top of Parquet files, turning a plain file directory into a reliable, queryable table.

**The transaction log** (`_delta_log/`) is the core of Delta Lake:
- A sequence of JSON files, one per committed transaction
- Each entry records which files were added and removed
- Readers compute the active file set from the log — no directory listing needed
- Checkpoints compress the log every 10 commits

**ACID guarantees:**
- *Atomicity* — a write commits all files or none
- *Consistency* — schema enforcement rejects invalid data at write time
- *Isolation* — optimistic concurrency; readers always see a consistent snapshot
- *Durability* — committed transactions are permanent

**Schema enforcement** — Delta rejects extra columns, wrong types, or missing non-nullable columns. Enable `mergeSchema` for intentional additive schema changes (new columns only).

**Time travel** — read any past version with `versionAsOf` or `timestampAsOf`. Every transaction is permanently preserved in the log (until `VACUUM` removes old files).

**DESCRIBE HISTORY** — full audit trail of every operation: operation type, timestamp, parameters, metrics.

**CONVERT TO DELTA** — upgrade an existing Parquet table to Delta in place, zero data copied.

The next notebook (ACID & Time Travel) covers MERGE, UPDATE, DELETE, rollback, and retention policies in depth.

In [ ]:
# Cleanup
spark.sql("DROP TABLE IF EXISTS customers_delta")
spark.sql("DROP TABLE IF EXISTS card_txns_delta")

if os.path.exists(SCRATCH):
    shutil.rmtree(SCRATCH)
    print(f"Removed: {SCRATCH}")
print("Cleanup complete.")